# AE2: Manejo de archivos CSV, JSON y TXT

Este notebook coordina tres clases, cada una en su propio archivo `.py`:

| Clase | Archivo | Responsabilidad |
|---|---|---|
| `ManejadorCSV` | `manejador_csv.py` | Importar con `csv.reader`, modificar y guardar con `csv.writer` |
| `ManejadorJSON` | `manejador_json.py` | Importar con `json.load`, modificar y guardar con `json.dump` |
| `GeneradorResumenTXT` | `generador_resumen_txt.py` | Generar `salidas/resumen.txt` con los resultados reales |

Flujo:

```
Descarga manual → importar → modificar → guardar nuevo CSV y JSON
                → tercera clase → resumen TXT con URL de origen
```

Los archivos originales se descargaron **manualmente** de Hugging Face (sin `load_dataset` ni token) y están en `datos/originales/`; el notebook trabaja solo con esos archivos locales. Ejecuta las celdas en orden.

## 0. Preparar la ruta de trabajo

Todas las rutas son relativas a la carpeta `AE2`. Esta celda ubica esa carpeta (por si el notebook se abre desde otra ubicación) y la agrega a `sys.path` para poder importar las clases.

In [1]:
import os
import sys
from pathlib import Path

AE2 = Path.cwd().resolve()
if not (AE2 / "manejador_csv.py").exists():
    AE2 = next(p for p in [AE2, *AE2.parents] if (p / "AE2" / "manejador_csv.py").exists()) / "AE2"

os.chdir(AE2)
sys.path.insert(0, str(AE2))
print("Carpeta de trabajo:", Path.cwd())

Carpeta de trabajo: /workspaces/AD_2026Q4_grupo_2/AE2


## 1. URL de origen de los archivos

Ambos archivos vienen del dataset **VariantWise – Indian cars variants** de Hugging Face (licencia CC BY 4.0, datos de [VariantWise](https://variantwise.com)):

- **CSV** – catálogo de versiones de carros nuevos en India (marca, modelo, versión, motor, combustible, transmisión y precio en lakh): <https://huggingface.co/datasets/variantwise/indian-cars-variants/blob/main/data/catalogue.csv>
- **JSON** – impuestos de rodamiento y matrícula por estado y ciudad: <https://huggingface.co/datasets/variantwise/indian-cars-variants/blob/main/data/road-tax.json>

El JSON es un objeto; la lista de ciudades está en la clave `cities`, que es la que se modifica. Las URL se guardan en variables para enviarlas al resumen.

In [2]:
URL_CSV = "https://huggingface.co/datasets/variantwise/indian-cars-variants/blob/main/data/catalogue.csv"
URL_JSON = "https://huggingface.co/datasets/variantwise/indian-cars-variants/blob/main/data/road-tax.json"

RUTA_CSV_ORIGINAL = "datos/originales/catalogue.csv"
RUTA_JSON_ORIGINAL = "datos/originales/road-tax.json"

RUTA_CSV_MODIFICADO = "datos/modificados/catalogo_carros_modificado.csv"
RUTA_JSON_MODIFICADO = "datos/modificados/impuestos_ciudades_modificado.json"
RUTA_RESUMEN = "salidas/resumen.txt"

print("CSV :", URL_CSV)
print("JSON:", URL_JSON)

CSV : https://huggingface.co/datasets/variantwise/indian-cars-variants/blob/main/data/catalogue.csv
JSON: https://huggingface.co/datasets/variantwise/indian-cars-variants/blob/main/data/road-tax.json


## 2. Importar las clases CSV y JSON e inicializar sus objetos

In [3]:
from manejador_csv import ManejadorCSV
from manejador_json import ManejadorJSON

csv_carros = ManejadorCSV(RUTA_CSV_ORIGINAL, url_origen=URL_CSV)
json_ciudades = ManejadorJSON(RUTA_JSON_ORIGINAL, url_origen=URL_JSON, clave_registros="cities")
print("Objeto CSV :", type(csv_carros).__name__, "->", csv_carros.ruta_entrada)
print("Objeto JSON:", type(json_ciudades).__name__, "->", json_ciudades.ruta_entrada)

Objeto CSV : ManejadorCSV -> datos/originales/catalogue.csv
Objeto JSON: ManejadorJSON -> datos/originales/road-tax.json


## 3. Importar los archivos locales

`importar()` abre cada archivo con `with open(...)`; el CSV se lee con `csv.reader` y el JSON con `json.load`.

In [4]:
filas = csv_carros.importar()
print(f"CSV importado: {len(filas)} registros, {len(csv_carros.encabezados)} columnas")
print("Encabezados:", csv_carros.encabezados)
print()
for fila in csv_carros.vista_previa(3):
    print(fila[:19])

CSV importado: 1281 registros, 21 columnas
Encabezados: ['market', 'currency', 'priceStyle', 'priceBasis', 'brand', 'model', 'modelId', 'bodyStyle', 'seats', 'trim', 'trimId', 'variantId', 'variant', 'engine', 'engineId', 'fuel', 'transmission', 'gearbox', 'price', 'source', 'url']

['market', 'currency', 'priceStyle', 'priceBasis', 'brand', 'model', 'modelId', 'bodyStyle', 'seats', 'trim', 'trimId', 'variantId', 'variant', 'engine', 'engineId', 'fuel', 'transmission', 'gearbox', 'price']
['IN', 'INR', 'lakh', 'ex-showroom Delhi', 'Jeep', 'Compass', 'jeep-compass', 'suv', '5', 'Sport', 'sport', 'sport-mt', 'Sport Diesel MT', '2.0 Multijet II Diesel (4x2)', 'multijet-20-4x2', 'diesel', '6MT', 'manual', '17.99']
['IN', 'INR', 'lakh', 'ex-showroom Delhi', 'Jeep', 'Compass', 'jeep-compass', 'suv', '5', 'Longitude (O2)', 'longitude-o2', 'longitude-o2-mt', 'Longitude (O2) Diesel MT', '2.0 Multijet II Diesel (4x2)', 'multijet-20-4x2', 'diesel', '6MT', 'manual', '23.63']
['IN', 'INR', 'lakh', 

In [5]:
ciudades = json_ciudades.importar()
print(f"JSON importado: {len(ciudades)} ciudades (clave 'cities')")
print("Claves del objeto JSON:", list(json_ciudades.contenido.keys()))
print("Campos de cada ciudad:", list(ciudades[0].keys()))
print()
for c in json_ciudades.vista_previa(3):
    print({k: c[k] for k in ("id", "name", "stateId", "tier")})

JSON importado: 95 ciudades (clave 'cities')
Claves del objeto JSON: ['name', 'description', 'homepage', 'contact', 'license', 'attribution', 'dataAsOf', 'countryCode', 'counts', 'notes', 'national', 'states', 'cities']
Campos de cada ciudad: ['id', 'name', 'stateId', 'tier', 'registration']

{'id': 'mumbai', 'name': 'Mumbai', 'stateId': 'maharashtra', 'tier': 1}
{'id': 'pune', 'name': 'Pune', 'stateId': 'maharashtra', 'tier': 1}
{'id': 'nagpur', 'name': 'Nagpur', 'stateId': 'maharashtra', 'tier': 2}


## 4. Modificar cada archivo con los métodos de su clase (antes y después)

### 4.1 CSV

1. **Actualizar un valor**: el precio de la versión `sport-mt` (Jeep Compass Sport Diesel MT) pasa a `18.49` lakh.
2. **Agregar un registro**: una versión nueva de prueba.
3. **Eliminar registros**: se eliminan las versiones de la marca `Tesla`.

In [6]:
COL = csv_carros.encabezados.index
VERSION = "sport-mt"

def ver(filas):
    return [(f[COL("brand")], f[COL("model")], f[COL("variant")], f[COL("price")]) for f in filas]

print("ANTES")
print("Registros:", len(csv_carros.filas))
print("Versión a actualizar:", ver(csv_carros.buscar("variantId", VERSION)))
print("Versiones Tesla:", ver(csv_carros.buscar("brand", "Tesla")))

ANTES
Registros: 1281
Versión a actualizar: [('Jeep', 'Compass', 'Sport Diesel MT', '17.99')]
Versiones Tesla: [('Tesla', 'Model Y', 'Premium Electric AT', '50.89'), ('Tesla', 'Model Y L', 'Premium Electric AT', '61.99')]


In [7]:
csv_carros.actualizar_valor("variantId", VERSION, "price", "18.49")

csv_carros.agregar_registro({
    "market": "IN", "currency": "INR", "priceStyle": "lakh", "priceBasis": "ex-showroom Delhi",
    "brand": "Grupo 2", "model": "Demo EV", "modelId": "grupo2-demo-ev", "bodyStyle": "hatchback",
    "seats": "5", "trim": "Base", "trimId": "base", "variantId": "grupo2-base-ev",
    "variant": "Base EV AT", "engine": "Motor eléctrico 30 kWh", "engineId": "ev-30",
    "fuel": "electric", "transmission": "1AT", "gearbox": "automatic", "price": "9.99",
    "source": "Grupo 2 (registro de prueba)", "url": "",
})

csv_carros.eliminar_registros("brand", "Tesla")

print("DESPUÉS")
print("Registros:", len(csv_carros.filas))
print("Versión actualizada:", ver(csv_carros.buscar("variantId", VERSION)))
print("Versiones Tesla:", ver(csv_carros.buscar("brand", "Tesla")))
print("Registro agregado:", ver(csv_carros.buscar("brand", "Grupo 2")))
print()
print("Modificaciones registradas:")
for m in csv_carros.modificaciones:
    print(" -", m)

DESPUÉS
Registros: 1280
Versión actualizada: [('Jeep', 'Compass', 'Sport Diesel MT', '18.49')]
Versiones Tesla: []
Registro agregado: [('Grupo 2', 'Demo EV', 'Base EV AT', '9.99')]

Modificaciones registradas:
 - Actualizar valor: variantId=sport-mt -> price: '17.99' => '18.49'
 - Agregar registro: IN | INR | lakh | ex-showroom Delhi | Grupo 2 | Demo EV | grupo2-demo-ev | hatchback | 5 | Base | base | grupo2-base-ev | Base EV AT | Motor eléctrico
 - Eliminar registros: brand=Tesla (2 filas)


### 4.2 JSON

Se modifica la lista `cities`:

1. **Actualizar un valor**: la ciudad `nagpur` pasa de categoría (`tier`) 2 a 1.
2. **Agregar un registro**: la ciudad `nellore` (Andhra Pradesh).
3. **Eliminar un registro**: la ciudad `leh`.

In [8]:
def ciudad(id_):
    return [{k: c[k] for k in ("id", "name", "stateId", "tier")} for c in json_ciudades.buscar("id", id_)]

print("ANTES")
print("Ciudades:", len(json_ciudades.datos))
print("nagpur :", ciudad("nagpur"))
print("nellore:", ciudad("nellore"))
print("leh    :", ciudad("leh"))

ANTES


Ciudades: 95
nagpur : [{'id': 'nagpur', 'name': 'Nagpur', 'stateId': 'maharashtra', 'tier': 2}]
nellore: []
leh    : [{'id': 'leh', 'name': 'Leh', 'stateId': 'ladakh', 'tier': 1}]


In [9]:
json_ciudades.actualizar_valor("id", "nagpur", "tier", 1)

json_ciudades.agregar_registro({
    "id": "nellore",
    "name": "Nellore",
    "stateId": "andhra-pradesh",
    "tier": 2,
    "registration": {},
})

json_ciudades.eliminar_registro("id", "leh")

print("DESPUÉS")
print("Ciudades:", len(json_ciudades.datos))
print("nagpur :", ciudad("nagpur"))
print("nellore:", ciudad("nellore"))
print("leh    :", ciudad("leh"))
print()
print("Modificaciones registradas:")
for m in json_ciudades.modificaciones:
    print(" -", m)

DESPUÉS
Ciudades: 95
nagpur : [{'id': 'nagpur', 'name': 'Nagpur', 'stateId': 'maharashtra', 'tier': 1}]
nellore: [{'id': 'nellore', 'name': 'Nellore', 'stateId': 'andhra-pradesh', 'tier': 2}]
leh    : []

Modificaciones registradas:
 - Actualizar valor: id=nagpur -> tier: 2 => 1
 - Agregar registro: {'id': 'nellore', 'name': 'Nellore', 'stateId': 'andhra-pradesh', 'tier': 2, 'registration': {}}
 - Eliminar registro: id='leh' (1 registros)


## 5. Guardar un nuevo CSV y un nuevo JSON y verificar los cambios

Los archivos se guardan con **nombres nuevos** en `datos/modificados/`, así los originales se conservan. Luego se vuelven a leer con objetos nuevos para comprobar que los cambios quedaron guardados.

In [10]:
ruta_csv = csv_carros.guardar(RUTA_CSV_MODIFICADO)
ruta_json = json_ciudades.guardar(RUTA_JSON_MODIFICADO)
print("CSV guardado en :", ruta_csv)
print("JSON guardado en:", ruta_json)

CSV guardado en : datos/modificados/catalogo_carros_modificado.csv
JSON guardado en: datos/modificados/impuestos_ciudades_modificado.json


In [11]:
# Verificación: leer los archivos nuevos con objetos nuevos
verif_csv = ManejadorCSV(RUTA_CSV_MODIFICADO)
verif_csv.importar()
verif_json = ManejadorJSON(RUTA_JSON_MODIFICADO, clave_registros="cities")
verif_json.importar()

print("CSV  modificado:", len(verif_csv.filas), "registros | original:", csv_carros.registros_importados)
print("  Precio de", VERSION, "=", verif_csv.buscar("variantId", VERSION)[0][COL("price")])
print("  Registro 'Grupo 2' existe:", bool(verif_csv.buscar("brand", "Grupo 2")))
print("  Versiones Tesla:", len(verif_csv.buscar("brand", "Tesla")))
print()
print("JSON modificado:", len(verif_json.datos), "ciudades | original:", json_ciudades.registros_importados)
print("  tier de nagpur:", verif_json.buscar("id", "nagpur")[0]["tier"])
print("  nellore existe:", bool(verif_json.buscar("id", "nellore")))
print("  leh existe:", bool(verif_json.buscar("id", "leh")))
print("  Se conservan las demás claves:", list(verif_json.contenido.keys()))

CSV  modificado: 1280 registros | original: 1281
  Precio de sport-mt = 18.49
  Registro 'Grupo 2' existe: True
  Versiones Tesla: 0

JSON modificado: 95 ciudades | original: 95
  tier de nagpur: 1
  nellore existe: True
  leh existe: False
  Se conservan las demás claves: ['name', 'description', 'homepage', 'contact', 'license', 'attribution', 'dataAsOf', 'countryCode', 'counts', 'notes', 'national', 'states', 'cities']


## 6. Generar el resumen `.txt` con la tercera clase

`GeneradorResumenTXT` recibe los resultados reales de las dos clases (incluidas las mismas URL de origen definidas en el paso 1) y escribe `salidas/resumen.txt`.

In [12]:
from generador_resumen_txt import GeneradorResumenTXT

resumen = GeneradorResumenTXT(RUTA_RESUMEN)
resumen.agregar_resultado(csv_carros.resultados())
resumen.agregar_resultado(json_ciudades.resultados())
ruta_resumen = resumen.guardar()

print("Resumen guardado en:", ruta_resumen)
print()
print(resumen.leer())

Resumen guardado en: salidas/resumen.txt

RESUMEN AE2 - MANEJO DE ARCHIVOS CSV, JSON Y TXT
Fecha de generación: 2026-10-24 02:37:21

1. URL de origen
------------------------------------------------------------
- CSV: https://huggingface.co/datasets/variantwise/indian-cars-variants/blob/main/data/catalogue.csv
- JSON: https://huggingface.co/datasets/variantwise/indian-cars-variants/blob/main/data/road-tax.json

2. Archivos importados
------------------------------------------------------------
- CSV: datos/originales/catalogue.csv (1281 registros)
- JSON: datos/originales/road-tax.json (clave 'cities') (95 registros)

3. Modificaciones realizadas
------------------------------------------------------------
CSV:
  - Actualizar valor: variantId=sport-mt -> price: '17.99' => '18.49'
  - Agregar registro: IN | INR | lakh | ex-showroom Delhi | Grupo 2 | Demo EV | grupo2-demo-ev | hatchback | 5 | Base | base | grupo2-base-ev | Base EV AT | Motor eléctrico
  - Eliminar registros: brand=Tesla 

## Conclusión

- Se importaron un CSV (catálogo de carros) y un JSON (impuestos por ciudad) descargados de Hugging Face, usando `csv.reader` y `json.load` dentro de `with open(...)`.
- Cada formato se modificó con métodos de su propia clase (actualizar, agregar y eliminar registros).
- Los cambios se guardaron con `csv.writer` y `json.dump` en archivos nuevos, conservando los originales.
- La tercera clase generó `salidas/resumen.txt` con las URL de origen, archivos, modificaciones, clases, métodos y cantidad de registros.